# Results Analysis
Loads `checkpoint_evals.json` for all conditions and plots training trajectories.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

ROOT = Path().resolve().parent
RESULTS = ROOT / 'results'

CONDITIONS = ['flat', 'chunked', 'balanced']
LABELS = {'flat': 'Sliding-Window', 'chunked': 'EOS-Chunked', 'balanced': 'DENSITY-Balanced'}
COLORS = {'flat': '#E64B35', 'chunked': '#4DBBD5', 'balanced': '#00A087'}

def load_evals(cond):
    p = RESULTS / cond / 'checkpoint_evals.json'
    if not p.exists(): return []
    data = json.loads(p.read_text())
    ckpts = [c for c in data.get('checkpoints', []) if c.get('words_seen')]
    return sorted(ckpts, key=lambda c: c['words_seen'])

evals = {c: load_evals(c) for c in CONDITIONS}
for c, v in evals.items():
    print(f'{c}: {len(v)} checkpoints')

## Final checkpoint numbers

In [ ]:
rows = []
for cond in CONDITIONS:
    ckpts = evals[cond]
    if not ckpts: continue
    last = ckpts[-1]
    blimp_ckpts = [c for c in ckpts if 'blimp' in c]
    rows.append({
        'Condition':   LABELS[cond],
        'Words Seen':  f"{last['words_seen']/1e6:.1f}M",
        'FBT Overall': round(last.get('fbt_overall', float('nan')), 3),
        'False Belief':round(last.get('fbt_false_belief', float('nan')), 3),
        'True Belief': round(last.get('fbt_true_belief', float('nan')), 3),
        'BLiMP':       round(blimp_ckpts[-1]['blimp'], 3) if blimp_ckpts else 'N/A',
        'EWoK':        round([c for c in ckpts if 'ewok' in c][-1]['ewok'], 3) if any('ewok' in c for c in ckpts) else 'N/A',
    })
pd.DataFrame(rows).set_index('Condition')

## FBT trajectories — False Belief Trap

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
fig.suptitle('FBT Accuracy During Training', fontsize=13, fontweight='bold')

for ax, cond in zip(axes, CONDITIONS):
    ckpts = evals[cond]
    if not ckpts:
        ax.set_title(f'{LABELS[cond]}\n(no data)')
        continue
    xs  = np.array([c['words_seen'] / 1e6 for c in ckpts])
    fbt = np.array([c['fbt_overall']       for c in ckpts], dtype=float)
    fb  = np.array([c['fbt_false_belief']  for c in ckpts], dtype=float)
    tb  = np.array([c['fbt_true_belief']   for c in ckpts], dtype=float)

    ax.axhline(0.5, color='#999', lw=1, ls='--', label='Chance')
    ax.plot(xs, fbt, color='#3C5488', lw=2,   label='Overall')
    ax.plot(xs, fb,  color='#E64B35', lw=1.5, label='False Belief')
    ax.plot(xs, tb,  color='#4DBBD5', lw=1.5, label='True Belief')

    ax.set_title(LABELS[cond], fontsize=11, fontweight='bold')
    ax.set_xlabel('Words Seen (M)')
    ax.set_ylim(0.10, 1.0)
    for sp in ['top','right']: ax.spines[sp].set_visible(False)

axes[0].set_ylabel('P(Correct)')
axes[0].legend(fontsize=9)
plt.tight_layout()
plt.show()

## BLiMP — syntactic competence

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.axhline(0.5, color='#999', lw=1, ls='--', label='Chance')
ax.axhline(0.6636, color='#333', lw=1, ls=':', label='BabyLM Baseline (66.36%)')

for cond in CONDITIONS:
    ckpts = [c for c in evals[cond] if 'blimp' in c]
    if not ckpts: continue
    xs = np.array([c['words_seen'] / 1e6 for c in ckpts])
    ys = np.array([c['blimp']            for c in ckpts], dtype=float)
    ax.plot(xs, ys, color=COLORS[cond], lw=2, label=LABELS[cond])

ax.set_xlabel('Words Seen (M)')
ax.set_ylabel('BLiMP Accuracy')
ax.set_title('BLiMP Over Training', fontsize=12, fontweight='bold')
ax.set_ylim(0.45, 0.75)
for sp in ['top','right']: ax.spines[sp].set_visible(False)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## EWoK — world knowledge (null result)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.axhline(0.5, color='#999', lw=1, ls='--', label='Chance')

DASH = {'flat': '-', 'chunked': '--', 'balanced': ':'}
for cond in CONDITIONS:
    ckpts = [c for c in evals[cond] if 'ewok' in c]
    if not ckpts: continue
    xs = np.array([c['words_seen'] / 1e6 for c in ckpts])
    ys = np.array([c['ewok']             for c in ckpts], dtype=float)
    ax.plot(xs, ys, color=COLORS[cond], lw=2, ls=DASH[cond], label=LABELS[cond])

ax.set_xlabel('Words Seen (M)')
ax.set_ylabel('EWoK Accuracy')
ax.set_title('EWoK Over Training — all conditions at chance', fontsize=12, fontweight='bold')
ax.set_ylim(0.44, 0.56)
for sp in ['top','right']: ax.spines[sp].set_visible(False)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## False Belief Bias (FB − TB gap)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.axhline(0, color='#999', lw=1, ls='--', label='No bias')

for cond in ['flat', 'chunked']:  # balanced has too few checkpoints
    ckpts = evals[cond]
    if not ckpts: continue
    xs  = np.array([c['words_seen'] / 1e6 for c in ckpts])
    gap = np.array([c['fbt_false_belief'] - c['fbt_true_belief'] for c in ckpts], dtype=float)
    ax.plot(xs, gap, color=COLORS[cond], lw=2, label=LABELS[cond])

ax.set_xlabel('Words Seen (M)')
ax.set_ylabel('FB − TB')
ax.set_title('False Belief Bias Over Training', fontsize=12, fontweight='bold')
for sp in ['top','right']: ax.spines[sp].set_visible(False)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Per-item breakdown — final checkpoint

In [ ]:
# Condition × Cue type breakdown from checkpoint_evals
rows = []
for cond in CONDITIONS:
    ckpts = evals[cond]
    if not ckpts: continue
    last = ckpts[-1]
    for key, val in last.items():
        if key.startswith('fbt_') and isinstance(val, float):
            rows.append({'condition': LABELS[cond], 'metric': key.replace('fbt_',''), 'value': round(val, 3)})

df = pd.DataFrame(rows)
df.pivot(index='metric', columns='condition', values='value')